# 02 — Exact Matching

Exact matching is the first building block: rows match when the values in the rule columns are **equal** (after standardization). We **start with raw** data and **standardize** it explicitly (same pipeline as 01), then match. So: raw → clean → match—no hidden steps. Here we run a single rule (`full_name`), then *cascading* rules (full_name, then email_clean for unmatched), and see how to evaluate results. The next notebook makes the **measurement loop** explicit.

**Back:** [01 Preparation](01_preparation.ipynb) · **Next:** [03 The Measurement Loop](03_measurement_loop.ipynb)

## 1. Load raw data, standardize, then match

We start with **raw** data (same as 01: column name mismatches, nulls, messy values), then **explicitly** standardize it so we have clean tables for matching. No magic: load raw → standardize → match.

**Data:** 500 left, 500 right, **30 known pairs**. After standardization we match on `full_name` and optionally cascade to `email_clean`.

In [4]:
import sys
from pathlib import Path
import polars as pl
from matcher import Matcher

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = (
        Path.cwd()
        if (Path.cwd() / "tutorial_data").exists()
        else Path.cwd().parent.parent / "docs" / "tutorial"
    )
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_entity_resolution, standardize_for_matching

# 1. Load raw (same as 01: right has email_address, postal_code; messy values)
left_raw, right_raw, ground_truth = load_entity_resolution()
print("Raw: right columns", right_raw.columns[:4], "...")

# 2. Standardize: align schema, add email_clean and full_name
left, right = standardize_for_matching(left_raw, right_raw)
print("Clean: same columns?", set(left.columns) == set(right.columns))

# 3. Match on clean data
matcher = Matcher(
    left=left,
    right=right,
    left_id="id",
    right_id="id",
)
n_left, n_right = left.shape[0], right.shape[0]
n_gt = ground_truth.shape[0]
print(f"Left: {n_left} rows, Right: {n_right} rows, Ground truth: {n_gt} pairs")

Raw: right columns ['id', 'email_address', 'first_name', 'last_name'] ...
Clean: same columns? True
Left: 500 rows, Right: 500 rows, Ground truth: 30 pairs


## 2. Single rule vs cascading rules

Start with **one rule** and see how many pairs we find. Then **add a second rule** (cascade): matcher tries the first rule, then for rows that didn't match it tries the second. Each step gives incrementally better recall.

**Step 2a:** Match on `full_name` only. We find the pairs that share the same full name.

Ground truth has columns `left_id` and `right_id`; the match table has `id` and `id_right`. We pass `left_id_col` and `right_id_col` so the evaluator can join them.

In [ ]:
results_name = matcher.match(on="full_name")
m_name = results_name.evaluate(
    ground_truth,
    left_id_col="id",
    right_id_col="id_right",
)
print(f"Matches: {results_name.count}")
tp, total = m_name["true_positives"], ground_truth.shape[0]
print(f"Recall: {m_name['recall']:.2%} (found {tp} of {total} true pairs)")

Matches: 10
Recall: 33.33% (found 10 of 30 true pairs)


**Step 2b:** Add `email_clean` as a **cascade** (second rule). Use `refine(on=...)`: first `match(on="full_name")`, then `.refine(on="email_clean")`. The second rule runs only on left rows that didn't match the first. (To match when *both* fields match in one step—AND logic—use a single rule: `match(on=["full_name", "email_clean"])`.)

In [6]:
# Cascade: first rule match(on="full_name"), then refine(on="email_clean") for unmatched rows
results_cascade = matcher.match(on="full_name").refine(on="email_clean")
m_cascade = results_cascade.evaluate(
    ground_truth,
    left_id_col="id",
    right_id_col="id_right",
)
print(f"Matches: {results_cascade.count}")
tp, total = m_cascade["true_positives"], ground_truth.shape[0]
print(f"Recall: {m_cascade['recall']:.2%} (found {tp} of {total} true pairs)")

ValueError: Only a single rule is allowed per match(). For cascading (next rule on unmatched), use .refine(on=...) after match().

## 3. Compare the two runs

We already evaluated each run above. Here's the full picture side by side: precision, recall, F1. The cascade adds recall (more true pairs found) while we keep an eye on precision. In the next notebook we'll turn this into a repeatable **measurement loop**.

In [ ]:
comparison = pl.DataFrame({
    "run": ["full_name only", "full_name+email"],
    "precision": [f"{m_name['precision']:.2%}", f"{m_cascade['precision']:.2%}"],
    "recall": [f"{m_name['recall']:.2%}", f"{m_cascade['recall']:.2%}"],
    "f1": [f"{m_name['f1']:.2%}", f"{m_cascade['f1']:.2%}"],
})
display(comparison)

run,precision,recall,f1
str,str,str,str
"""full_name only""","""100.00%""","""33.33%""","""50.00%"""
"""full_name+email""","""100.00%""","""66.67%""","""80.00%"""


You've seen the evaluate API. Next we'll discuss how to **use these evaluation metrics** and **understand the tradeoffs** (e.g. when to favour recall vs precision, how to tune). That's [03 The Measurement Loop](03_measurement_loop.ipynb)—measure → change one thing → measure again → compare. After that we add fuzzy matching (04).